In [ ]:
import requests
import pandas as pd
import numpy as np
from pathlib import Path
import time
from datetime import datetime, timezone

# Project paths
PROJECT_ROOT = Path.home() / "wspr-propagation"
WSPR_DATA_DIR = PROJECT_ROOT / "data" / "wspr"
WSPR_DATA_DIR.mkdir(parents=True, exist_ok=True)

# WSPR live endpoint
WSPR_URL = "https://db1.wspr.live/"

# Bands of interest (in meters)
BANDS = [10, 20, 40]

# Geographic bounding box — North America + Europe
# tx or rx must fall within this box
LAT_MIN, LAT_MAX = 25.0, 70.0
LON_MIN, LON_MAX = -130.0, 40.0

print(f"Data directory: {WSPR_DATA_DIR}")
print(f"Bands: {BANDS}m")

In [ ]:
def fetch_wspr_day(date: str, band: int) -> pd.DataFrame:
    """
    Fetch one day of WSPR spots for a given band.
    date: 'YYYY-MM-DD'
    band: band code (28=10m, 14=20m, 7=40m)
    """
    
    query = f"""
        SELECT
            time,
            tx_sign,
            rx_sign,
            tx_loc,
            rx_loc,
            tx_lat,
            tx_lon,
            rx_lat,
            rx_lon,
            distance,
            band,
            frequency,
            power,
            snr,
            drift
        FROM wspr.rx
        WHERE
            date(time) = '{date}'
            AND band = {band}
            AND tx_lat BETWEEN {LAT_MIN} AND {LAT_MAX}
            AND tx_lon BETWEEN {LON_MIN} AND {LON_MAX}
            AND rx_lat BETWEEN {LAT_MIN} AND {LAT_MAX}
            AND rx_lon BETWEEN {LON_MIN} AND {LON_MAX}
        FORMAT JSONCompact
    """
    
    try:
        response = requests.get(
            WSPR_URL,
            params={"query": query},
            timeout=60
        )
        response.raise_for_status()
        
        data = response.json()
        
        if not data.get("data"):
            print(f"  No data returned for {date} band={band}")
            return pd.DataFrame()
        
        cols = [col["name"] for col in data["meta"]]
        df = pd.DataFrame(data["data"], columns=cols)
        
        df["time"] = pd.to_datetime(df["time"])
        for col in ["tx_lat","tx_lon","rx_lat","rx_lon","frequency","snr","drift"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        df["power"] = pd.to_numeric(df["power"], errors="coerce")
        df["distance"] = pd.to_numeric(df["distance"], errors="coerce")
        df["band"] = pd.to_numeric(df["band"], errors="coerce")
        
        return df
    
    except Exception as e:
        print(f"  Error fetching {date} band={band}: {e}")
        return pd.DataFrame()

# Band code mapping
BAND_CODES = {10: 28, 20: 14, 40: 7}

print("fetch_wspr_day() redefined")
print(f"Band codes: {BAND_CODES}")

In [ ]:
from datetime import date

# Gannon storm window — May 7-14, 2024
# Day 7-9: pre-storm, flare buildup
# Day 10: storm onset 17:00 UTC
# Day 11: Kp=9 peak
# Day 12: recovery begins
# Day 13-14: late recovery + X8.7 flare on May 14

gannon_dates = [date(2024, 5, d) for d in range(7, 15)]
BANDS = [10, 20, 40]
BAND_CODES = {10: 28, 20: 14, 40: 7}

print("Gannon storm fetch schedule:")
for d in gannon_dates:
    print(f"  {d.strftime('%Y-%m-%d')} ({d.strftime('%A')})")
print(f"\nTotal: {len(gannon_dates)} days × {len(BANDS)} bands = {len(gannon_dates)*len(BANDS)} fetches")

In [ ]:
import time

def fetch_all(schedule: list, bands_m: list, delay: float = 2.0) -> None:
    """
    Fetch and cache all days and bands in the schedule.
    Skips already-cached files automatically.
    delay: seconds between requests to be polite to wspr.live
    """
    total = len(schedule) * len(bands_m)
    completed = 0
    skipped = 0
    failed = 0

    for d in schedule:
        date_str = d.strftime("%Y-%m-%d")
        for band_m in bands_m:
            path = day_parquet_path(date_str, band_m)
            if path.exists():
                skipped += 1
                completed += 1
                continue
            try:
                fetch_and_cache_day(date_str, band_m)
                time.sleep(delay)
            except Exception as e:
                print(f"  FAILED {date_str} {band_m}m: {e}")
                failed += 1

            completed += 1
            remaining = total - completed
            pct = 100 * completed / total
            print(f"  [{pct:.0f}%] {completed}/{total} done, {skipped} skipped, {failed} failed, {remaining} remaining")

print("fetch_all() defined")

In [ ]:
def day_parquet_path(date: str, band_m: int) -> Path:
    """Return the parquet path for a given date and band in meters."""
    return WSPR_DATA_DIR / f"wspr_{date}_{band_m}m.parquet"

# Band code mapping
BAND_CODES = {10: 28, 20: 14, 40: 7}

print("day_parquet_path() defined")

In [ ]:
# Dry run
print("Dry run — checking cache status:")
already_cached = 0
to_fetch = 0
for d in gannon_dates:
    date_str = d.strftime("%Y-%m-%d")
    for band_m in BANDS:
        path = day_parquet_path(date_str, band_m)
        if path.exists():
            already_cached += 1
        else:
            to_fetch += 1

print(f"  Already cached: {already_cached}")
print(f"  To fetch:       {to_fetch}")
print(f"  Estimated time: {to_fetch * 45 / 60:.0f} minutes")

In [ ]:
def fetch_wspr_day(date: str, band: int) -> pd.DataFrame:
    query = f"""
        SELECT time, tx_sign, rx_sign, tx_loc, rx_loc,
               tx_lat, tx_lon, rx_lat, rx_lon, distance,
               band, frequency, power, snr, drift
        FROM wspr.rx
        WHERE date(time) = '{date}'
          AND band = {band}
          AND tx_lat BETWEEN {LAT_MIN} AND {LAT_MAX}
          AND tx_lon BETWEEN {LON_MIN} AND {LON_MAX}
          AND rx_lat BETWEEN {LAT_MIN} AND {LAT_MAX}
          AND rx_lon BETWEEN {LON_MIN} AND {LON_MAX}
        FORMAT JSONCompact
    """
    try:
        response = requests.get(WSPR_URL, params={"query": query}, timeout=60)
        response.raise_for_status()
        data = response.json()
        if not data.get("data"):
            return pd.DataFrame()
        cols = [col["name"] for col in data["meta"]]
        df = pd.DataFrame(data["data"], columns=cols)
        df["time"] = pd.to_datetime(df["time"])
        for col in ["tx_lat","tx_lon","rx_lat","rx_lon","frequency","snr","drift"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        df["power"] = pd.to_numeric(df["power"], errors="coerce")
        df["distance"] = pd.to_numeric(df["distance"], errors="coerce")
        df["band"] = pd.to_numeric(df["band"], errors="coerce")
        return df
    except Exception as e:
        print(f"  Error fetching {date} band={band}: {e}")
        return pd.DataFrame()

def day_parquet_path(date: str, band_m: int) -> Path:
    return WSPR_DATA_DIR / f"wspr_{date}_{band_m}m.parquet"

def fetch_and_cache_day(date: str, band_m: int, force: bool = False) -> pd.DataFrame:
    path = day_parquet_path(date, band_m)
    if path.exists() and not force:
        return pd.read_parquet(path)
    band_code = BAND_CODES.get(band_m)
    if band_code is None:
        raise ValueError(f"Unknown band: {band_m}m")
    print(f"  Fetching {date} {band_m}m ...", end=" ")
    df = fetch_wspr_day(date, band_code)
    if df.empty:
        print("empty.")
        return df
    df = df[df["distance"] > 0]
    df = df[df["snr"].between(-35, 20)]
    df = df[df["power"].between(0, 57)]
    df = df[df["drift"].abs() <= 4]
    df["path_loss_proxy"] = df["power"] - df["snr"]
    df.to_parquet(path, index=False)
    print(f"{len(df):,} rows cached.")
    return df

BAND_CODES = {10: 28, 20: 14, 40: 7}

print("All fetch functions defined.")

In [ ]:
# Fetch all Gannon storm dates
fetch_all(gannon_dates, BANDS, delay=2.0)
print("\nFetch complete.")

In [ ]:
# Audit the Gannon fetch
print("Gannon fetch audit:")
for d in gannon_dates:
    date_str = d.strftime("%Y-%m-%d")
    row = []
    for band_m in BANDS:
        path = day_parquet_path(date_str, band_m)
        if path.exists():
            df_check = pd.read_parquet(path)
            row.append(f"{band_m}m:{len(df_check):,}")
        else:
            row.append(f"{band_m}m:MISSING")
    print(f"  {date_str}: {' | '.join(row)}")